# Notebook 3 — Model Training

**Purpose:** Train binary classifiers to detect cheaters from engineered player features.

**Input:** `raw.player_features_engineered` (PostgreSQL) or `data/processed/player_features_engineered.csv` (fallback)

**Models trained:**
| Model | Notes |
|---|---|
| Logistic Regression | Baseline linear classifier |
| Random Forest | `n_estimators=100`, ensemble method |
| XGBoost | Gradient boosting, typically best performer |

**Output:** Three `.pkl` model files saved to `src/`

## Cell 1 — Imports & Configuration

In [ ]:
import os
import warnings
import pandas as pd
import joblib
from pathlib import Path
from dotenv import load_dotenv

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb

warnings.filterwarnings("ignore")
load_dotenv()

SRC_DIR = Path("../src")
SRC_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    "avg_kills", "avg_deaths", "kd_ratio",
    "avg_accuracy", "avg_headshot_rate", "avg_reaction_time_ms",
    "avg_damage_dealt", "matches_played",
    "stddev_accuracy", "stddev_reaction_time_ms",
    "accuracy_z_score", "reaction_z_score", "headshot_z_score",
    "suspicion_score", "consistency_flag",
]
TARGET_COL = "cheater_flag"
RANDOM_STATE = 42

print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

## Cell 2 — Load Data (PostgreSQL with CSV fallback)

In [ ]:
CSV_PATH = Path("../data/processed/player_features_engineered.csv")

db_url = os.environ.get("DATABASE_URL")
if db_url:
    try:
        from sqlalchemy import create_engine, text
        engine = create_engine(db_url)
        with engine.connect() as conn:
            df = pd.read_sql(
                text("SELECT * FROM raw.player_features_engineered"), conn
            )
        print(f"Loaded {len(df):,} rows from PostgreSQL")
    except Exception as exc:
        print(f"PostgreSQL unavailable ({exc}), falling back to CSV")
        df = pd.read_csv(CSV_PATH)
        print(f"Loaded {len(df):,} rows from {CSV_PATH}")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df):,} rows from {CSV_PATH}")

print(f"\nClass distribution:\n{df[TARGET_COL].value_counts().to_string()}")
print(f"\nCheater rate: {df[TARGET_COL].mean():.1%}")
df[FEATURE_COLS + [TARGET_COL]].head(3)

## Cell 3 — Train / Test Split

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows")
print(f"Train cheater rate: {y_train.mean():.1%}")
print(f"Test  cheater rate: {y_test.mean():.1%}")

# Persist test set so notebook 04 can reload it without re-splitting
test_df = X_test.copy()
test_df[TARGET_COL] = y_test.values
test_df.to_csv(SRC_DIR / "test_set.csv", index=False)
print("\nSaved test set → src/test_set.csv")

## Cell 4 — Model 1: Logistic Regression (Baseline)

In [ ]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")),
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("=" * 55)
print("  LOGISTIC REGRESSION — Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_lr, target_names=["Legit", "Cheater"]))

joblib.dump(lr_pipeline, SRC_DIR / "logistic_regression_model.pkl")
print("Saved → src/logistic_regression_model.pkl")

## Cell 5 — Model 2: Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("=" * 55)
print("  RANDOM FOREST — Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_rf, target_names=["Legit", "Cheater"]))

joblib.dump(rf_model, SRC_DIR / "random_forest_model.pkl")
print("Saved → src/random_forest_model.pkl")

## Cell 6 — Model 3: XGBoost

In [ ]:
# class_weight equivalent: scale_pos_weight = n_negatives / n_positives
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {neg}/{pos} = {scale_pos_weight:.1f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    use_label_encoder=False,
    verbosity=0,
)

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("\n" + "=" * 55)
print("  XGBOOST — Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_xgb, target_names=["Legit", "Cheater"]))

joblib.dump(xgb_model, SRC_DIR / "xgboost_model.pkl")
print("Saved → src/xgboost_model.pkl")

## Cell 7 — Summary

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

results = []
for name, y_pred in [
    ("Logistic Regression", y_pred_lr),
    ("Random Forest",       y_pred_rf),
    ("XGBoost",             y_pred_xgb),
]:
    results.append({
        "Model":     name,
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1":        round(f1_score(y_test, y_pred), 4),
    })

summary = pd.DataFrame(results).set_index("Model")
print("=" * 55)
print("  TRAINING SUMMARY (test set, cheater class)")
print("=" * 55)
print(summary.to_string())
print("\nAll models saved to src/")